# Step 1: Install and import required libraries

In [ ]:
!pip install pandas nltk tqdm openai
import pandas as pd
import json
import random
from tqdm import tqdm
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


# Step 2: Mount Google Drive to access datasets


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


# Step 3: Load and clean Amazon review datasets

In [ ]:
def load_dataset(file_path):
    data = []
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))

    df = pd.DataFrame(data)

    expected_cols = ['asin','reviewText','overall']
    cols_to_keep = [col for col in expected_cols if col in df.columns]
    df = df[cols_to_keep]

    if 'reviewText' in df.columns:
        df = df.dropna(subset=['reviewText'])

    return df


# Load datasets
beauty_df = load_dataset("/content/drive/MyDrive/ShopHallu/All_Beauty.json")
grocery_df = load_dataset("/content/drive/MyDrive/ShopHallu/Grocery_and_Gourmet_Food.json")
accessories_df = load_dataset("/content/drive/MyDrive/ShopHallu/Cell_Phones_and_Accessories.json")

print("Beauty reviews:", len(beauty_df))
print("Grocery reviews:", len(grocery_df))
print("Accessories reviews:", len(accessories_df))

Beauty reviews: 5264
Grocery reviews: 1143470
Accessories reviews: 1127672


# Step 4: Sample products and prepare evidence + generation text

In [ ]:
def sample_products(df, category_name, n_products=50):
    review_counts = df.groupby("asin").size()
    valid_products = review_counts[review_counts >= 8].index
    filtered_reviews = df[df["asin"].isin(valid_products)]
    n_products = min(len(valid_products), n_products)
    sample_list = random.sample(list(valid_products), n_products)

    products = []
    for asin in sample_list:
        product_reviews = filtered_reviews[filtered_reviews["asin"] == asin]
        reviewText = product_reviews["reviewText"].dropna().astype(str).tolist()
        random.shuffle(reviewText)

        evidence_reviews = reviewText[:5]
        generation_reviews = reviewText[5:8]

        products.append({
            "asin": asin,
            "category": category_name,
            "evidence_reviews": evidence_reviews,
            "generation_reviews": generation_reviews,
            "generation_text": " ".join(generation_reviews)
        })
    return pd.DataFrame(products)

# Step 5: Combine product samples across categories

In [ ]:
import random

# Sample products from beauty reviews
products_df = sample_products(beauty_df, "Beauty", n_products=33)

# Sample from grocery as well and concatenate
grocery_products_df = sample_products(grocery_df, "Grocery", n_products=33)

accessories_products_df = sample_products(accessories_df, "Accessories", n_products=33)

# CONCAT ALL DATA (total = 99)
products_df = pd.concat(
    [products_df, grocery_products_df, accessories_products_df],
    ignore_index=True
)

# BALANCED CONTROL: 33 per category
products_df = products_df.groupby("category").sample(n=33, random_state=42)

print(products_df.shape)
print(products_df["category"].value_counts())

print(products_df.head())

(99, 5)
category
Accessories    33
Beauty         33
Grocery        33
Name: count, dtype: int64
          asin     category  \
97  B018QQCKU2  Accessories   
81  B0093FHN8U  Accessories   
92  B004I8WD6E  Accessories   
83  B004BOWHZ2  Accessories   
74  B00OUIKHHY  Accessories   

                                     evidence_reviews  \
97  [This product reduces device clutter considera...   
81  [Love the little tools that came with it. That...   
92  [Very nice, clear iphone protector.  It stays ...   
83  [nice, Great case for this phone., Good for my...   
74  [The case is extremely protective, but unfortu...   

                                   generation_reviews  \
97  [works great and convenient storage for my IPO...   
81  [Watched a youtube video for how to replace my...   
92  [I love that this case is transparent since it...   
83  [Does the job and for the price I really canno...   
74  [I have this same case on my LG3. It has prove...   

                              

# Step 6: Setup OpenRouter API client

In [ ]:
from openai import OpenAI
import os

os.environ["OPENROUTER_API_KEY"] = "OPENROUTER_API_KEY"

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

# Step 7: Generate influencer-style posts using LLM

In [ ]:
import time

def generate_post(context, prompt_type, model_name, temp):

    if prompt_type == "strict":
        prompt = f"""
        You are writing an Instagram influencer post about a product.

        Use the following customer reviews as the source of information.
        Write a clear and natural one-paragraph post (4–6 sentences).
        Describe the product experience using the information from the reviews,
        but keep the tone informative and restrained, like a careful influencer
        summarizing what people say about the product.

        Reviews:
        {context}
        """

    elif prompt_type == "neutral":
        prompt = f"""
        Write a natural Instagram influencer post promoting this product.

        The post should be one paragraph with several sentences (4–6 sentences).
        Make it engaging and conversational like a typical influencer caption.
        Use ideas from the customer reviews but write it in your own words.

        Reviews:
        {context}
        """

    elif prompt_type == "persuasive":
        prompt = f"""
        Write a highly persuasive Instagram influencer post promoting this product.

        The post should be one paragraph with multiple sentences (4–6 sentences).
        Emphasize benefits, excitement, and positive experiences the product may
        provide. Use an enthusiastic influencer tone and highlight why people
        might love the product.

        Reviews:
        {context}
        """


    for attempt in range(2):

        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=temp,
                timeout=60
            )

            return response.choices[0].message.content

        except Exception as e:

            print(f"{model_name} failed: {e}")

            time.sleep(2)

    return None

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
print(len(products_df))
print(products_df["category"].value_counts())
print(products_df.columns)

99
category
Accessories    33
Beauty         33
Grocery        33
Name: count, dtype: int64
Index(['asin', 'category', 'evidence_reviews', 'generation_text'], dtype='object')


In [ ]:
test_df = products_df.sample(3, random_state=42)

# Step 8: Expand into labeling dataset (paragraph + evidence pairs)

In [ ]:
from tqdm import tqdm
import pandas as pd
import hashlib

final_rows = []

models = [
    "openai/gpt-4o-mini",
    "google/gemini-3.5-flash",
    "meta-llama/llama-3.1-8b-instruct"
]

temperatures = [0.0, 0.3, 0.5, 0.7, 1.0]
prompt_types = ["strict", "neutral", "persuasive"]


cache = {}

for _, row in tqdm(products_df.iterrows(), total=len(products_df)):

    context = row["generation_text"]

    evidence_list = row.get("evidence_reviews", [])


    if isinstance(evidence_list, list) and len(evidence_list) > 0:
        evidence_list = evidence_list[:5]
    else:
        evidence_list = []

    for model_name in models:
        for temp in temperatures:
            for prompt_type in prompt_types:

                # CACHE KEY
                key = hashlib.md5(
                    (context + model_name + str(temp) + prompt_type).encode()
                ).hexdigest()

                if key in cache:
                    post = cache[key]
                else:
                    post = generate_post(
                        context=context,
                        prompt_type=prompt_type,
                        model_name=model_name,
                        temp=temp
                    )
                    cache[key] = post

                # skip failed generations
                if post is None:
                    continue

                # attach evidence AFTER generation (no extra API cost)
                if len(evidence_list) == 0:
                    final_rows.append({
                        "asin": row["asin"],
                        "category": row["category"],
                        "prompt_type": prompt_type,
                        "model": model_name,
                        "temperature": temp,
                        "influencer_review": post,
                        "evidence_review": None,
                        "supported": None,
                        "fabricated_feature": None,
                        "exaggerated_performance": None,
                        "contradiction": None,
                        "Unsupported Promotional Implication": None
                    })
                else:
                    for evidence in evidence_list:
                        final_rows.append({
                            "asin": row["asin"],
                            "category": row["category"],
                            "prompt_type": prompt_type,
                            "model": model_name,
                            "temperature": temp,
                            "influencer_review": post,
                            "evidence_review": evidence,
                            "supported": None,
                            "fabricated_feature": None,
                            "exaggerated_performance": None,
                            "contradiction": None,
                            "Unsupported Promotional Implication": None
                        })

final_df = pd.DataFrame(final_rows)

print(final_df.shape)
final_df.head()

 44%|████▍     | 44/99 [2:22:23<2:56:14, 192.26s/it]

meta-llama/llama-3.1-8b-instruct failed: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'meta-llama/llama-3.1-8b-instruct is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Groq', 'previous_errors': [{'code': 429, 'message': 'Provider returned error', 'provider_name': 'DeepInfra', 'raw': 'meta-llama/llama-3.1-8b-instruct is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations'}, {'code': 429, 'message': 'Provider returned error', 'provider_name': 'DeepInfra', 'raw': 'meta-llama/llama-3.1-8b-instruct is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations'}, {'code': 429, 'message': 'Provider returned error', 'provi

100%|██████████| 99/99 [5:21:45<00:00, 195.00s/it]


(22275, 12)


,asin,category,prompt_type,model,temperature,influencer_review,evidence_review,supported,fabricated_feature,exaggerated_performance,contradiction,emotional_only
0,B007JZU8N8,Accessories,strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,Screen protector is good but my was dead ..so..,None,None,None,None,None
1,B007JZU8N8,Accessories,strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,Provides required protection without observing...,None,None,None,None,None
2,B007JZU8N8,Accessories,strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,ok,None,None,None,None,None
3,B007JZU8N8,Accessories,strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,I was pleasantly surprised at the thickness of...,None,None,None,None,None
4,B007JZU8N8,Accessories,strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,Ok,None,None,None,None,None


In [ ]:
final_df.to_csv(
    "01.ShopHallu_dataset_generated_for_rule_based_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

# Additional Step: Recover Missing Generation Context (3-Review Set per Product) & Fix Row Duplication (22,275 → 4,455)

In [ ]:
import pandas as pd
import json

final_df = pd.read_excel("/content/drive/MyDrive/ShopHallu/01.ShopHallu_dataset_generated_for_rule_based_analysis.xlsx")

rows = []
for asin, sub in final_df.groupby("asin", sort=False):
    first_paragraph = sub.iloc[:5]
    ev5 = first_paragraph["evidence_review"].tolist()
    assert len(ev5) == 5, f"{asin} has {len(ev5)} evidence reviews, expected 5"
    rows.append({
        "asin": asin,
        "category": sub["category"].iloc[0],
        "evidence_reviews_known": ev5,
    })

known_df = pd.DataFrame(rows)
known_df["evidence_reviews_known"] = known_df["evidence_reviews_known"].apply(json.dumps)

print(known_df.shape)
print(known_df.head(3))

# ---- Save ----
known_df.to_csv("/content/drive/MyDrive/ShopHallu/01.ShopHallu_dataset_generated_for_rule_based_analysis_with_generation_text.csv", index=False)
print("Saved to Drive: known_evidence_by_asin.csv")

(99, 3)
         asin     category                             evidence_reviews_known
0  B007JZU8N8  Accessories  ["Screen protector is good but my was dead ..s...
1  B00HZ79SQ8  Accessories  ["Perfect fit, great protection, crystal clear...
2  B00CKUDPS2  Accessories  ["Great Product, works perfect.", "I really lo...
Saved to Drive: known_evidence_by_asin.csv


In [ ]:
import pandas as pd
import json
import random
from collections import Counter

# ---- Loading existing 22,275-row labeling file ----
final_df = pd.read_excel("/content/drive/MyDrive/ShopHallu/01.ShopHallu_dataset_generated_for_rule_based_analysis.xlsx")

# ---- Getting the 5 known evidence_review texts per ASIN ----
known_evidence = {}
for asin, sub in final_df.groupby("asin", sort=False):
    known_evidence[asin] = sub.iloc[:5]["evidence_review"].tolist()

# ---- Mapping category -> raw dataframe ----
category_to_df = {
    "Accessories": accessories_df,
    "Beauty": beauty_df,
    "Grocery": grocery_df,
}

# ----  3-review generation context per ASIN ----
random.seed(42)

context_rows = []
for asin, ev5 in known_evidence.items():
    category = final_df.loc[final_df["asin"] == asin, "category"].iloc[0]
    df = category_to_df[category]
    pool = df[df["asin"] == asin]["reviewText"].dropna().astype(str).tolist()

    pool_counter = Counter(pool)
    for ev in ev5:
        if pool_counter[ev] > 0:
            pool_counter[ev] -= 1
    candidates = list(pool_counter.elements())

    if len(candidates) == 3:
        chosen = candidates
    else:
        chosen = random.sample(candidates, 3)

    context_rows.append({
        "asin": asin,
        "generation_reviews": json.dumps(chosen),
        "generation_text": " ".join(chosen),
    })

context_df = pd.DataFrame(context_rows)


merged = final_df.merge(context_df, on="asin", how="left")

# insert generation_reviews / generation_text right after 'category',
cols = merged.columns.tolist()
cols.remove("generation_reviews")
cols.remove("generation_text")
cat_idx = cols.index("category")
new_cols = cols[:cat_idx+1] + ["generation_reviews", "generation_text"] + cols[cat_idx+1:]
merged = merged[new_cols]

assert len(merged) == len(final_df)  # confirm row count untouched (22,275)

# ---- Blank out repeated asin values within each group (merged-cell look) ----
merged["asin"] = merged["asin"].where(merged["asin"] != merged["asin"].shift(1), "")

print(merged.shape)
print(merged.head(10)[["asin", "category", "generation_text"]])

# ---- Save as xlsx ----
merged.to_excel("/content/drive/MyDrive/ShopHallu/01.ShopHallu_dataset_generated_for_rule_based_analysis_with_generation_text.xlsx", index=False)
print("Saved to Drive: ShopHallu_with_context_22275.xlsx")

(22275, 14)
         asin     category                                    generation_text
0  B007JZU8N8  Accessories  good to have, fits very well Expecting anti-gl...
1              Accessories  good to have, fits very well Expecting anti-gl...
2              Accessories  good to have, fits very well Expecting anti-gl...
3              Accessories  good to have, fits very well Expecting anti-gl...
4              Accessories  good to have, fits very well Expecting anti-gl...
5              Accessories  good to have, fits very well Expecting anti-gl...
6              Accessories  good to have, fits very well Expecting anti-gl...
7              Accessories  good to have, fits very well Expecting anti-gl...
8              Accessories  good to have, fits very well Expecting anti-gl...
9              Accessories  good to have, fits very well Expecting anti-gl...
Saved to Drive: ShopHallu_with_context_22275.xlsx
